In [1]:
import requests
import settings
import json
import os
import time
from typing import Dict, List, Optional, Any
from datetime import datetime

Definimos el modelo credencial primero porque nuestro agente ACApyClient lo usará para representar las credenciales emitidas o recibidas

In [2]:
class Credential:
    _json_file = "models_data.json"  # Mismo archivo que Model
    _credentials_key = "credentials"  # Key específica para credenciales
    
    def __init__(self, definition_id, credential_info, owner_did: str = None):
        self.id = definition_id
        self.info = credential_info
        self.owner_did = owner_did
        self.timestamp = self._get_timestamp()
        self.credential_id = self._generate_credential_id()
        
        # Auto-saving al crear la instancia
        self._save_to_json()

    def _generate_credential_id(self):
        """Genera un ID único para la credencial"""
        import uuid
        return f"cred_{uuid.uuid4().hex[:8]}"

    def _get_timestamp(self):
        """Obtiene timestamp actual"""
        return datetime.now().isoformat()

    def _save_to_json(self):
        """Guarda la credencial en el archivo JSON"""
        try:
            # Cargar datos existentes
            existing_data = self._load_existing_data()
            
            # Preparar datos de la credencial
            credential_data = {
                "credential_id": self.credential_id,
                "definition_id": self.id,
                "info": self.info,
                "owner_did": self.owner_did,
                "timestamp": self.timestamp,
                "type": self.__class__.__name__
            }
            
            # Agregar datos específicos si existen
            if hasattr(self, '_get_credential_data'):
                credential_data.update(self._get_credential_data())
            
            # Actualizar o crear la sección de credenciales
            if self._credentials_key not in existing_data:
                existing_data[self._credentials_key] = []
            
            # Verificar si ya existe esta credencial (por ID)
            existing_index = None
            for i, cred in enumerate(existing_data[self._credentials_key]):
                if cred.get('credential_id') == self.credential_id:
                    existing_index = i
                    break
            
            if existing_index is not None:
                # Actualizar credencial existente
                existing_data[self._credentials_key][existing_index] = credential_data
            else:
                # Agregar nueva credencial
                existing_data[self._credentials_key].append(credential_data)
            
            # Guardar en archivo
            self._save_data_to_file(existing_data)
            
        except Exception as e:
            print(f"❌ Error guardando credencial en JSON: {e}")

    def _load_existing_data(self) -> Dict:
        """Carga los datos existentes del archivo JSON"""
        if os.path.exists(self._json_file):
            try:
                with open(self._json_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                return {}
        return {}

    def _save_data_to_file(self, data: Dict):
        """Guarda los datos en el archivo JSON"""
        with open(self._json_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    def update_info(self, new_info: Dict):
        """Actualiza la información de la credencial y guarda automáticamente"""
        self.info.update(new_info)
        self.timestamp = self._get_timestamp()
        self._save_to_json()

    def delete(self):
        """Elimina la credencial del JSON"""
        try:
            existing_data = self._load_existing_data()
            
            if self._credentials_key in existing_data:
                # Filtrar la credencial a eliminar
                existing_data[self._credentials_key] = [
                    cred for cred in existing_data[self._credentials_key] 
                    if cred.get('credential_id') != self.credential_id
                ]
                
                self._save_data_to_file(existing_data)
                print(f"✅ Credencial {self.credential_id} eliminada")
                
        except Exception as e:
            print(f"❌ Error eliminando credencial: {e}")

    @classmethod
    def load_all_credentials(cls) -> List[Dict]:
        """Carga todas las credenciales del archivo JSON"""
        try:
            existing_data = cls._load_existing_data(cls)
            return existing_data.get(cls._credentials_key, [])
        except Exception as e:
            print(f"❌ Error cargando credenciales: {e}")
            return []

    @classmethod
    def find_by_owner_did(cls, did: str) -> List[Dict]:
        """Encuentra credenciales por DID del propietario"""
        all_credentials = cls.load_all_credentials()
        return [cred for cred in all_credentials if cred.get('owner_did') == did]

    @classmethod
    def find_by_definition_id(cls, definition_id: str) -> List[Dict]:
        """Encuentra credenciales por ID de definición"""
        all_credentials = cls.load_all_credentials()
        return [cred for cred in all_credentials if cred.get('definition_id') == definition_id]

    def to_dict(self) -> Dict:
        """Convierte la credencial a diccionario"""
        return {
            "credential_id": self.credential_id,
            "definition_id": self.id,
            "info": self.info,
            "owner_did": self.owner_did,
            "timestamp": self.timestamp,
            "type": self.__class__.__name__
        }

La clase ACApyClient simulará ser nuestro servidor que tiene el rol más alto dentro de nuestra red Indy (Von Netwok)

In [3]:
class ACApyClient:
    def __init__(self, wallet_token: Optional[str] = None):
        self.admin_url = settings.ACA_PY_CONFIG['admin_url']
        self.seeder = "V4SGRU86Z58d6TV7PBUe6f"
        self.headers = {
            'Content-Type': 'application/json',
            'Accept': 'application/json',
        }
        if wallet_token:
            self.headers['Authorization'] = f'Bearer {wallet_token}'

    def create_wallet(self, wallet_name: str, wallet_key: str, label: str = None):
        """Crea un nuevo wallet en modo multitenant"""
        url = f"{self.admin_url}/multitenancy/wallet"
        payload = {
            "wallet_name": wallet_name,
            "wallet_key": wallet_key,
            "label": label or wallet_name,
            "wallet_type": "askar",
            "key_management_mode": "managed",
            "seed": self.seeder
        }
        
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_wallet_token(self, wallet_id: str, wallet_key: str):
        """Obtiene token de acceso para un wallet específico"""
        url = f"{self.admin_url}/multitenancy/wallet/{wallet_id}/token"
        payload = {
            "wallet_key": wallet_key
        }
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        return response.json()['token']

    def delete_wallet(self, wallet_id: str):
        """Elimina un wallet multitenant"""
        url = f"{self.admin_url}/multitenancy/wallet/{wallet_id}"
        response = requests.delete(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def create_local_did(self):
        """Crea un DID local dentro del wallet del agente"""
        url = f"{self.admin_url}/wallet/did/create"
        payload = {"method": "sov", "options": {"key_type": "ed25519"}}
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        return response.json().get("result", {})

    def register_did_in_ledger(self, did, verkey):
        """Registrar el DID en el ledger de Indy"""
        
        # PRIMERO: Configurar el DID como público en el wallet del tenant
        set_public_url = f"{self.admin_url}/wallet/did/public"
        set_public_payload = {
            "did": did
        }
        
        print(f"📤 Configurando DID como público: {did}")
        public_response = requests.post(set_public_url, params=set_public_payload, headers=self.headers)
        
        print(f"📥 Response configurar público: {public_response.status_code}")
        if public_response.status_code != 200:
            print(f"❌ Error configurando DID público: {public_response.text}")
            # Continuar de todas formas, a veces igual funciona
        
        # LUEGO: Registrar en el ledger
        url = f"{self.admin_url}/ledger/register-nym"
        payload = {
            "did": did,
            "verkey": verkey,
            "alias": "AriesCLI",
            "role": "ENDORSER"
        }

        print(f"📤 Enviando request a: {url}")
        print(f"📝 Payload: {json.dumps(payload, indent=2)}")

        # CORRECCIÓN: Usar json= en lugar de params=
        response = requests.post(url, params=payload, headers=self.headers)
        
        print(f"📥 Status: {response.status_code}")
        print(f"📥 Body: {response.text}")

        if response.ok:
            try:
                return response.json()
            except json.JSONDecodeError:
                return {"raw_response": response.text}
        else:
            return {
                "error": f"HTTP {response.status_code}",
                "raw_response": response.text
            }

    def register_wallet(self):
        """Crea un DID local y lo registra en el ledger"""
        did_info = self.create_local_did()
        did = did_info["did"]
        verkey = did_info["verkey"]
        ledger_response = self.register_did_in_ledger(did, verkey)
        return {"did": did, "verkey": verkey, "ledger_tx": ledger_response}

    # ------------------------
    # 📜 Schemas y CredDefs
    # ------------------------
    def register_credential_schema(self, name, version, attributes):
        """Registra un nuevo schema en el ledger"""
        schema_payload = {
            "schema_name": name,
            "schema_version": version,
            "attributes": attributes
        }
        url = f"{self.admin_url}/schemas"
        response = requests.post(url, json=schema_payload, headers=self.headers)
        response.raise_for_status()
        result = response.json()
        return result.get("schema", result.get("sent", {}))

    def create_credential_definition(self, schema_id, tag="default", support_revocation=False):
        """Crea una credential definition basada en un schema existente"""
        url = f"{self.admin_url}/credential-definitions"
        payload = {
            "schema_id": schema_id,
            "tag": tag,
            "support_revocation": support_revocation
        }
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        data = response.json()
        return data.get("credential_definition_id")

    def create_credential(self, name, version, attributes, support_revocation=False):        
        try:
            # Registrar schema
            schema_response = self.register_credential_schema(name, version, attributes)
            
            if schema_response:
                # Crear credential definition
                cred_def_id = self.create_credential_definition(
                    schema_id=schema_response['id'],
                    support_revocation=support_revocation
                )
                return Credential(cred_def_id, schema_response)
            return None
            
        except Exception as e:
            print(f"❌ Error creando credencial: {e}")
            return None

    def get_existing_schemas(self):
        """Obtiene todos los schemas existentes del ledger"""
        try:
            url = f"{self.admin_url}/schemas/created"
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json().get('schema_ids', [])
        except Exception as e:
            print(f"⚠️ Error obteniendo schemas existentes: {e}")
            return []
    
    def schema_exists(self, schema_name: str, schema_version: str) -> bool:
        """Verifica rápidamente si un schema ya existe"""
        existing_schemas = self.get_existing_schemas()
        
        # Buscar en la lista de schema_ids
        target_pattern = f"{schema_name}/{schema_version}"
        for schema_id in existing_schemas:
            if target_pattern in schema_id:
                return True
        return False
    
    def get_existing_cred_defs(self):
        """Obtiene todas las credential definitions existentes"""
        try:
            url = f"{self.admin_url}/credential-definitions/created"
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json().get('credential_definition_ids', [])
        except Exception as e:
            print(f"⚠️ Error obteniendo cred defs existentes: {e}")
            return []
    
    def get_cred_def_by_schema(self, schema_id: str):
        """Obtiene credential definition por schema_id"""
        cred_defs = self.get_existing_cred_defs()
        print(f"🔍 Cred Defs disponibles: {cred_defs}")

        for cred_def_id in cred_defs:
            if schema_id in cred_def_id:
                print(f"✅ Cred Def encontrada: {cred_def_id}")
                return cred_def_id
            
        print(f"❌ No se encontró Cred Def para schema: {schema_id}")
        return None
    
    # ------------------------
    # 🔗 Conexiones
    # ------------------------
    def create_invitation(self):
        """Crea una invitación de conexión"""
        url = f"{self.admin_url}/connections/create-invitation"
        response = requests.post(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def accept_invitation(self, invitation_url: str):
        """Acepta una invitación de conexión"""
        url = f"{self.admin_url}/connections/receive-invitation"
        # Extraer el payload de la URL de invitación
        invitation_payload = self._parse_invitation_url(invitation_url)
        response = requests.post(url, json=invitation_payload, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def _parse_invitation_url(self, invitation_url: str):
        """Parsea una URL de invitación en payload JSON"""
        # Implementar parsing de URL de invitación
        # Esto es un ejemplo simplificado
        import base64
        import json as json_lib
        
        if invitation_url.startswith('http'):
            # Es una URL completa, extraer el parámetro de invitación
            from urllib.parse import urlparse, parse_qs
            parsed = urlparse(invitation_url)
            query_params = parse_qs(parsed.query)
            if 'c_i' in query_params:
                invitation_b64 = query_params['c_i'][0]
                invitation_json = base64.urlsafe_b64decode(invitation_b64 + '==')
                return json_lib.loads(invitation_json)
        
        # Si ya es JSON, devolver como está
        try:
            return json_lib.loads(invitation_url)
        except:
            raise ValueError("Formato de invitación no válido")

    # ------------------------
    # 🎓 Emisión de Credenciales
    # ------------------------
    def send_credential_offer(self, cred_def_id, attributes):
        """
        Envía una oferta de credencial usando una cred_def existente.
        attributes: lista de dicts [{"name": "campo", "value": "valor"}]
        """
        invitation = self.create_invitation()
        connection_id = invitation["connection_id"]

        offer_payload = {
            "connection_id": connection_id,
            "cred_def_id": cred_def_id,
            "credential_preview": {
                "@type": "issue-credential/1.0/credential-preview",
                "attributes": attributes
            },
            "auto_issue": True,
            "auto_remove": True
        }

        print(f"📤 Enviando oferta de credencial: {json.dumps(offer_payload, indent=2)}")

        url = f"{self.admin_url}/issue-credential/send-offer"
        response = requests.post(url, json=offer_payload, headers=self.headers)
        print(f"📥 Status: {response.status_code}")
        print(f"📥 Body: {response.text}")

        if response.ok:
            return response.json()
        else:
            return {
                "error": f"HTTP {response.status_code}",
                "raw_response": response.text
            }

    def get_credential_offers(self):
        """Obtiene todas las ofertas de credenciales pendientes"""
        url = f"{self.admin_url}/issue-credential/records"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json().get('results', [])

    def accept_credential_offer(self, cred_ex_id: str):
        """Acepta una oferta de credencial"""
        url = f"{self.admin_url}/issue-credential/records/{cred_ex_id}/send-request"
        response = requests.post(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def store_credential(self, cred_ex_id: str):
        """Almacena una credencial recibida"""
        url = f"{self.admin_url}/issue-credential/records/{cred_ex_id}/store"
        response = requests.post(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_credentials(self):
        """Obtiene todas las credenciales almacenadas"""
        url = f"{self.admin_url}/credentials"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json().get('results', [])

    def get_credential_by_id(self, credential_id: str):
        """Obtiene una credencial específica por ID"""
        url = f"{self.admin_url}/credentials/{credential_id}"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

La clase ACApyClientTenant simulará ser nuestros cliente, cada instancia creada será un cliente que tiene el rol más bajo dentro de nuestra red Indy (Von Netwok)

In [4]:
class ACApyClientTenant:
    def __init__(self, wallet_token: str):
        self.admin_url = settings.ACA_PY_CONFIG['admin_url']
        self.seeder = "V4SGRU86Z58d6TV7PBUe6f"
        self.headers = {
            'Content-Type': 'application/json',
            'Accept': 'application/json',
            'Authorization': f'Bearer {wallet_token}',
        }
        self.base_client = ACApyClient()
        self.did = None
        self.verkey = None

    def register_wallet(self):
        public_did = self.base_client.create_local_did()
        response = self.register_did_in_ledger(public_did['did'], public_did['verkey'])
        print(response)
        return response

    def register_did_in_ledger(self, did, verkey):
        """Registrar el DID en el ledger usando el agente base"""
        url = f"{self.base_client.admin_url}/ledger/register-nym"
        params = {
            "did": did,
            "verkey": verkey,
            "alias": "AriesCLI",
            "role": "ENDORSER"
        }
        response = requests.post(url, params=params, headers=self.base_client.headers)
        if response.ok:
            self.did = did
            self.verkey= verkey
            return response.json()
        else:
            return {"error": response.text, "status": response.status_code}
    
    def get_credentials(self):
        """
        Obtiene todas las credenciales almacenadas en el wallet del tenant.
        """
        try:
            url = f"{self.admin_url}/credentials"
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            
            data = response.json()
            credentials = data.get('results', [])
            
            print(f"✅ {len(credentials)} credenciales obtenidas del wallet")
            return credentials

        except Exception as e:
            print(f"❌ Error obteniendo credenciales: {e}")
            return []
    
    def get_credential_offers(self):
        """
        Obtiene todas las ofertas de credenciales pendientes (state=offer-received)
        desde el agente Aries.
        """
        url = f"{self.admin_url}/issue-credential-2.0/records?state=offer-received"

        response = requests.get(url, headers=self.headers)

        if response.status_code != 200:
            raise Exception(f"Error al obtener ofertas de credenciales: {response.text}")

        data = response.json()
        return data.get("results", [])

In [5]:
class Wallet:
    def __init__(self, wallet_name: Optional[str] = None):
        self.wallet_name = wallet_name or f"wallet_{self._generate_id()}"
        self.wallet_key = self._generate_wallet_key()
        self.wallet_id = None
        self.wallet_token = None
        self.client = None
        self.wallet_data = {}
        
        self._initialize_wallet()

    def _initialize_wallet(self):
        """Inicializa el wallet en ACA-Py"""
        try:
            # Crear cliente base (sin token)
            base_client = ACApyClient()
            
            # Crear wallet multitenant
            wallet_info = base_client.create_wallet(
                wallet_name=self.wallet_name,
                wallet_key=self.wallet_key,
                label=self.wallet_name
            )
            
            self.wallet_id = wallet_info['wallet_id']
            
            # Obtener token de acceso
            self.wallet_token = base_client.get_wallet_token(
                self.wallet_id, 
                self.wallet_key
            )
            
            # Crear cliente autenticado
            self.client = ACApyClientTenant(self.wallet_token)
            
            # Registrar DID
            wallet_data = self.client.register_wallet()
            self.wallet_data = wallet_data
            self.wallet_data['wallet_id'] = self.wallet_id
            self.wallet_data['wallet_name'] = self.wallet_name
            
        except Exception as e:
            print(f"❌ Error inicializando wallet {self.wallet_name}: {e}")
            raise

    def _generate_wallet_key(self):
        """Generar clave segura para el wallet"""
        import secrets
        import string
        alphabet = string.ascii_letters + string.digits
        return ''.join(secrets.choice(alphabet) for _ in range(32))

    def _generate_id(self):
        """Generar ID único"""
        import uuid
        return str(uuid.uuid4())[:8]

    def accept_credential_invitation(self, invitation_url: str):
        """Acepta una invitación de credencial"""
        return self.client.accept_invitation(invitation_url)

    def get_pending_credential_offers(self):
        """Obtiene ofertas de credenciales pendientes"""
        return self.client.get_credential_offers()

    def accept_all_pending_credentials(self):
        """Acepta y almacena todas las credenciales pendientes"""
        offers = self.get_pending_credential_offers()
        stored_credentials = []
        
        for offer in offers:
            if offer['state'] == 'offer_received':
                try:
                    # Aceptar oferta
                    cred_ex_id = offer['cred_ex_id']
                    self.client.accept_credential_offer(cred_ex_id)
                    
                    # Almacenar credencial
                    stored_cred = self.client.store_credential(cred_ex_id)
                    stored_credentials.append(stored_cred)
                    
                except Exception as e:
                    print(f"❌ Error aceptando credencial {offer['cred_ex_id']}: {e}")
        
        return stored_credentials

    def get_stored_credentials(self):
        """Obtiene todas las credenciales almacenadas"""
        return self.client.get_credentials()

# CREACIÓN DE SCHEMAS PARA LAS CREDENCIALES
En caso de no tener ningun esquema ejecutar esta parte

In [6]:
client = ACApyClient()
evtol_credential = {
    "name":"Evtol_Crede",
    "version": "3.0",
    "schema": ["id_puerto", "updates", "status"]
    }
user_credential = {
    "name":"User_Cred",
    "version": "1.0",
    "schema": ["first_name", "last_name", "can_ride"]
}
vertiport_credential = {
    "name":"Port_Credential",
    "version": "4.0",
    "schema": ["last_name", "n_airstrip", "n_parkings", "coord_lon","coord_lat"]
}

*Las siguientes credenciales son el ejemplo de como deben ser creadas, pero se usarán en la clase de Model, por lo que en un futuro es necesario ponerlas en un archivo JSON y ejecutarlo al comienzo de la aplicación para que puedan ser cargadas y su DID reconocido al crear las clases*

### CREACIÓN DE UNA CREDENCIAL DE TIPO EVTOL
Llamando POR SEPARADO a los métodos de:
- Registrar schema
- Crear definición de credencial

In [7]:
# Registramos este esquema en el ledger, en este caso el esqueña de una credencial evtol
temp_evtol_credential_schema = client.register_credential_schema(name=evtol_credential['name'], version=evtol_credential['version'], attributes=evtol_credential['schema'])
temp_evtol_credential_schema

{'ver': '1.0',
 'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:3.0',
 'name': 'Evtol_Crede',
 'version': '3.0',
 'attrNames': ['updates', 'id_puerto', 'status'],
 'seqNo': 7}

In [8]:
temp_evtol_credential_schema["id"]

'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:3.0'

In [9]:
# Creamos la definición de credencial del equema que acabamos de registrar
temp_evtol_credential_definition_id = client.create_credential_definition(schema_id=temp_evtol_credential_schema['id'], support_revocation=False)
temp_evtol_credential_definition_id

'V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default'

### CREACIÓN DE UNA CREDENCIAL DE TIPO USER
Llamando JUNTO a los métodos de:
- Registrar schema
- Crear definición de credencial

Mediante el método create_credential

Los mismos 2 pasos que se hizo en el caso del evtol, solo que dentro de uno solo (puede llamarse a este método en ves de los 2 anteriores, el resultado será el mismo al final el ledger registrará el esquema y luego la definición de credencial)

Sin embargo este método (a diferencia de usar los otros 2 por separado) devovlerá un objeto de tipo Credencial mientras que el otro solo te devuelve el id, aunque ambos cumplen con guardarlo en el ledger

In [10]:
temp_user_credential = client.create_credential(user_credential['name'], user_credential['version'], user_credential['schema'])
temp_user_credential.id

'V4SGRU86Z58d6TV7PBUe6f:3:CL:9:default'

In [11]:
# Aquí solo guardamos la variable porque después será usada por la clase Model, como se explico esto solo es de prueba...
# en un futuro esto tiene que ejecutarse al deplegar la red para tener la definiciones listas (obviamente después de ya haber declarado la clase ACApyCLient)
temp_user_credential_definition_id = temp_user_credential.id
temp_user_credential_definition_id

'V4SGRU86Z58d6TV7PBUe6f:3:CL:9:default'

### CREACIÓN DE UNA CREDENCIAL DE TIPO VERTIPORT
Llamando JUNTO a los métodos de:
- Registrar schema
- Crear definición de credencial

Mediante el método create_credential

Como ya sabemos este método devuelve un objeto de tipo credential por lo que podemos imprimir otros atributos como en este caso es info (esto no se podía hacer cuando llamamos los métodos por separado proque solo devolvía el id)

In [12]:
temp_vertiport_credential =  client.create_credential(vertiport_credential['name'], vertiport_credential['version'], vertiport_credential['schema'])
temp_vertiport_credential.info

{'ver': '1.0',
 'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Port_Credential:4.0',
 'name': 'Port_Credential',
 'version': '4.0',
 'attrNames': ['n_airstrip',
  'n_parkings',
  'coord_lon',
  'coord_lat',
  'last_name'],
 'seqNo': 11}

In [13]:
temp_vertiport_credential.info["id"]

'V4SGRU86Z58d6TV7PBUe6f:2:Port_Credential:4.0'

In [14]:
# Igual que las credenciales de los user, esto es solo para la prueba
temp_vertiport_credential_definition_id = temp_vertiport_credential.id
temp_vertiport_credential_definition_id

'V4SGRU86Z58d6TV7PBUe6f:3:CL:11:default'

# DEFINICIÓN DE UN CLASE BASE (MODEL) PARA LAS ENTIDADES

In [15]:
class Model:
    _instances: Dict[str, List[Dict[str, Any]]] = {}
    _json_file = "models_data.json"
    
    def __init__(self, id=None, wallet_name: Optional[str] = None):
        self.id = id if id is not None else self._generate_id()
        self.wallet = Wallet(wallet_name)
        self.did = self.wallet.client.did
        self.verkey = self.wallet.client.verkey
        self.token = self.wallet.wallet_token
        self.credentials = []
        self.entity_type = self.__class__.__name__.lower()
        
        # Registrar instancia
        self._register_instance()
        
        # Crear credencial automáticamente
        self._create_entity_credential()
        
        # Cargar credenciales existentes
        self._load_credentials()
        
        # Aceptar cualquier credencial pendiente automáticamente
        self._accept_pending_credentials()

    def _create_entity_credential(self):
        """Crea automáticamente la credencial correspondiente al tipo de entidad"""

        try:
            issuer_client = ACApyClient()

            # TO DO: Esto es para validar el uso de esquemas de credenciales que ya se encuentren en el ledger, pero lo mejor
            # sería que esto se encuentre en un archivo JSON que se genera al comenzar el programa después de declarar la clase ACApyClient
            # por ahora esta usado de esta manera, pero es algo que mejorar para un futuro algo tipo:
            # with open("credential_definitions.json") as f:
            #     cred_definitions = json.load(f)

            # if isinstance(self, Evtol):
            #     cred_def_id = cred_definitions["evtol"]
            # if isinstance(self, User):
            # ...
            cred_definitions = {
                "user": temp_evtol_credential_definition_id,
                "evtol": temp_evtol_credential_definition_id,
                "vertiport": temp_evtol_credential_definition_id
            }

            if self.entity_type not in cred_definitions:
                print(f"⚠️ Tipo de entidad '{self.entity_type}' sin definición registrada")
                return

            cred_def_id = cred_definitions[self.entity_type]
            
            # attributes obtiene un diccionario con los valores a incluir en la credencial, el método _get_credential_attributes
            # está definido en cada tipo de entidad, mantiniendo este método genérico
            if hasattr(self, "_get_credential_attributes"):
                attributes = self._get_credential_attributes()
            else:
                print(f"⚠️ La clase hija {self.entity_type} no define _get_credential_attributes()")
                return
            
            # SOLICITUD PARA LA CREACIÓN DE UNA INVITACIÓN DE CONEXIÓN
            # Creación de un diccionario JSON con información básica sobre el holder (el que recibirá la credencial)
            conn_data = {
                "their_label": self.wallet.wallet_name,
                "their_role": "holder"
            }
            # Apunta al endpoint del agente emisor
            conn_url = f"{issuer_client.admin_url}/connections/create-invitation"
            # Se hace un POST con la info básica del holder
            conn_response = requests.post(conn_url, json=conn_data, headers=issuer_client.headers)
            conn_response.raise_for_status()
            connection = conn_response.json()
            # connection_id es el identificador local de la conexión
            connection_id = connection["connection_id"]

            # CONSTRUCCIÓN DE LA CREDENCIAL
            credential_data = {
                "connection_id": connection_id,
                "credential_definition_id": cred_def_id,
                "credential_preview": {
                    "@type": "issue-credential/2.0/credential-preview",
                    "attributes": attributes
                },
                "auto_remove": False,
                "trace": False
            }

            # EMISIÓN DE LA CREDENCIAL
            # Endpoint por el que el issuer se encargará de emitir la credencial
            cred_url = f"{issuer_client.admin_url}/issue-credential-2.0/send"

            print("\n=== DEBUG: Datos de la credencial a enviar ===")
            print(json.dumps(credential_data, indent=4))
            print("=============================================\n")

            # Envía una solicitud HTTP POST al endpoint con todos los datos de la credencial
            cred_response = requests.post(cred_url, json=credential_data, headers=issuer_client.headers)
            cred_response.raise_for_status()

            result = cred_response.json()
            print(f"✅ Credencial '{self.entity_type}' emitida correctamente y enviada al holder {self.wallet.wallet_name}")

            # GUARDAR EL REGISTRO
            self.credentials.append(result)

            time.sleep(3)

            # Consultar credenciales en el holder
            holder_client = self.wallet.client  # ya autenticado con el token del holder
            creds = holder_client.get("/credentials")

            if creds.get("results"):
                print(f"✅ El holder {self.wallet.wallet_name} ahora tiene {len(creds['results'])} credencial(es) guardadas.")
            else:
                print(f"⚠️ No se encontraron credenciales en la wallet del holder.")

        except Exception as e:
            print(f"❌ Error emitiendo credencial para {self.wallet.wallet_name}: {e}")
        
    def _get_credential_attributes(self):
        """Obtiene los atributos específicos para la credencial - DEBE SER IMPLEMENTADO POR SUBCLASES"""
        # Este método debe ser sobrescrito por cada subclase
        return []

    def _generate_id(self):
        """Generar ID único para el modelo"""
        import uuid
        return str(uuid.uuid4())

    def _register_instance(self):
        """Registra la instancia en el JSON"""
        class_name = self.__class__.__name__
        if class_name not in Model._instances:
            Model._instances[class_name] = []
        
        instance_data = {
            "id": self.id,
            "did": self.did,
            "verkey": self.verkey,
            "token": self.token,
            "wallet_data": self.wallet.wallet_data,
            "credentials": self.credentials,
            "wallet_name": self.wallet.wallet_name,
            "timestamp": self._get_timestamp()
        }
        
        # Agregar datos específicos de la clase hija
        if hasattr(self, '_get_instance_data'):
            instance_data.update(self._get_instance_data())
        
        Model._instances[class_name].append(instance_data)
        self._save_to_json()

    def _save_to_json(self):
        """Guarda todas las instancias en JSON"""
        import json as json_lib
        import os
        
        try:
            existing_data = {}
            if os.path.exists(Model._json_file):
                with open(Model._json_file, 'r', encoding='utf-8') as f:
                    existing_data = json_lib.load(f)
            
            # Actualizar con datos actuales
            for class_name, instances in Model._instances.items():
                existing_data[class_name] = instances
            
            with open(Model._json_file, 'w', encoding='utf-8') as f:
                json_lib.dump(existing_data, f, indent=2, ensure_ascii=False)
                
        except Exception as e:
            print(f"❌ Error guardando en JSON: {e}")

    def _get_timestamp(self):
        """Obtiene timestamp actual"""
        from datetime import datetime
        return datetime.now().isoformat()

    def _load_credentials(self):
        """Carga credenciales almacenadas del wallet"""
        try:
            stored_creds = self.wallet.get_stored_credentials()
            self.credentials = stored_creds
        except Exception as e:
            print(f"⚠️ Error cargando credenciales: {e}")
            self.credentials = []

    def _accept_pending_credentials(self):
        """Acepta automáticamente credenciales pendientes"""
        try:
            new_credentials = self.wallet.accept_all_pending_credentials()
            if new_credentials:
                print(f"✅ {self.__class__.__name__} {self.id} aceptó {len(new_credentials)} credenciales")
                self._load_credentials()  # Recargar lista de credenciales
                self._register_instance()  # Actualizar en JSON
        except Exception as e:
            print(f"⚠️ Error aceptando credenciales pendientes: {e}")

    def accept_credential_invitation(self, invitation_url: str):
        """Acepta una invitación de credencial"""
        result = self.wallet.accept_credential_invitation(invitation_url)
        # Procesar credenciales pendientes después de aceptar invitación
        self._accept_pending_credentials()
        return result

    def get_credentials_info(self):
        """Obtiene información de las credenciales"""
        return {
            "total_credentials": len(self.credentials),
            "credentials": self.credentials
        }

In [16]:
# A partir de Model se puede crear la entidad de tipo Evtol
class Evtol(Model):
    def __init__(self, id=None, wallet_name=None, model="Standard", max_speed=120, id_puerto="default", status="active"):
        self.model = model
        self.max_speed = max_speed
        self.id_puerto = id_puerto
        self.status = status
        super().__init__(id, wallet_name)
    
    def _get_instance_data(self):
        return {
            "model": self.model,
            "max_speed": self.max_speed,
            "id_puerto": self.id_puerto,
            "status": self.status,
            "type": "eVTOL"
        }

    def _get_credential_attributes(self):
        """Implementa los atributos específicos para la credencial de Evtol"""
        return [
            {"name": "id_puerto", "value": self.id_puerto},
            {"name": "updates", "value": "0"},  # Inicialmente 0 updates
            {"name": "status", "value": self.status}
        ]

In [17]:
class User(Model):
    def __init__(self, id=None, wallet_name=None, first_name="", last_name="", can_ride=True):
        self.first_name = first_name
        self.last_name = last_name
        self.can_ride = can_ride
        super().__init__(id, wallet_name)
    
    def _get_instance_data(self):
        return {
            "first_name": self.first_name,
            "last_name": self.last_name,
            "can_ride": self.can_ride,
            "type": "User"
        }
    
    def _get_credential_attributes(self):
        return [
            {"name": "first_name", "value": self.first_name},
            {"name": "last_name", "value": self.last_name},
            {"name": "can_ride", "value": str(self.can_ride).lower()}
        ]

In [18]:
class Vertiport(Model):
    def __init__(self, id=None, wallet_name=None, name="", n_airstrip=1, n_parkings=5, coord_lon=0.0, coord_lat=0.0):
        self.name = name
        self.n_airstrip = n_airstrip
        self.n_parkings = n_parkings
        self.coord_lon = coord_lon
        self.coord_lat = coord_lat
        super().__init__(id, wallet_name)
    
    def _get_instance_data(self):
        return {
            "name": self.name,
            "n_airstrip": self.n_airstrip,
            "n_parkings": self.n_parkings,
            "coord_lon": self.coord_lon,
            "coord_lat": self.coord_lat,
            "type": "Vertiport"
        }
    
    def _get_credential_attributes(self):
        return [
            {"name": "last_name", "value": self.name},  # Usamos 'name' como 'last_name' en el schema
            {"name": "n_airstrip", "value": str(self.n_airstrip)},
            {"name": "n_parkings", "value": str(self.n_parkings)},
            {"name": "coord_lon", "value": str(self.coord_lon)},
            {"name": "coord_lat", "value": str(self.coord_lat)}
        ]

# REGISTRAR ENTIDADES EN EL LEDGER
Cada entidad que se cree a partir de esta clase base (Model), será registrado en el ledger de indy con sus propios did, token y wallets

In [19]:
temp_evtol = Evtol(
    model="Volocopter V2",
    max_speed=150,
    id_puerto="BER-001",
    status="active"
)

{'success': True}

=== DEBUG: Datos de la credencial a enviar ===
{
    "connection_id": "aa500348-e531-4a80-bef7-aced3eb54bcd",
    "credential_definition_id": "V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default",
    "credential_preview": {
        "@type": "issue-credential/2.0/credential-preview",
        "attributes": [
            {
                "name": "id_puerto",
                "value": "BER-001"
            },
            {
                "name": "updates",
                "value": "0"
            },
            {
                "name": "status",
                "value": "active"
            }
        ]
    },
    "auto_remove": false,
    "trace": false
}

❌ Error emitiendo credencial para wallet_e7ceabdd: 422 Client Error: Unprocessable Entity for url: http://localhost:9031/issue-credential-2.0/send
✅ 0 credenciales obtenidas del wallet


# PRUEBA: EMISIÓN DE CREDENCIALES
En este caso seguiremos el flujo de Evtol para poder arreglar posibles bugs

El proceos que queremos seguir es

1. Invite      → issuer
2. Request     → holder
3. Response    → issuer
4. Complete    → holder

In [20]:
# Primero creamos una wallet de ejemplo que usaremos para pruebas
temp_wallet = Wallet()
print("DID ya registrado y funcional para el Indy : ", temp_wallet.client.did)
print("Verkey del DID: ", temp_wallet.client.verkey)

{'success': True}
DID ya registrado y funcional para el Indy :  4id3iK6oqFvTDBKKmLUXEX
Verkey del DID:  32XDnxnJKwDMvbryB4y9Xqt4TM4kK5HtHfcMxf9Sc8T7


In [21]:
# Iniciamos el issuer_client y definimos la credential definition del evtol
# Como evtol ha sido creado distinto entonces por esta prueba tendra una pequeña variación,
# pero en una siguiente actualización será igual a los demás

temp_entity_type = "evtol"
temp_issuer_client = ACApyClient()

temp_cred_definitions = {
    "user": temp_user_credential.id,
    "evtol": temp_evtol_credential_definition_id,
    "vertiport": temp_vertiport_credential.id
}

temp_schemas_ids = {
    "user": temp_user_credential.info["id"],
    "evtol": temp_evtol_credential_schema["id"],
    "vertiport": temp_vertiport_credential.info["id"]
}

if temp_entity_type not in temp_cred_definitions:
    print(f"⚠️ Tipo de entidad '{temp_entity_type}' sin definición registrada")

temp_cred_def_id = temp_cred_definitions[temp_entity_type]
temp_schema_id = temp_schemas_ids[temp_entity_type]

print("Credential definition a usar: ", temp_cred_def_id)
print("Schema id a usar: ", temp_schema_id)

Credential definition a usar:  V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default
Schema id a usar:  V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:3.0


In [22]:
temp_cred_def_id

'V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default'

In [23]:
# Ahora vamos con la creacion de atributos que definen la clase hija, pero lo haremos manual ya que no usaremos las clases
# Lo haremos con el caso de evtol

# if hasattr(self, "_get_credential_attributes"):
#     attributes = self._get_credential_attributes()
# else:
#     print(f"⚠️ La clase hija {self.entity_type} no define _get_credential_attributes()")
#     return

temp_attributes = [
    {"name": "id_puerto", "value": "BER-001"},
    {"name": "updates", "value": "0"},
    {"name": "status", "value": "active"}    
]
temp_attributes

[{'name': 'id_puerto', 'value': 'BER-001'},
 {'name': 'updates', 'value': '0'},
 {'name': 'status', 'value': 'active'}]

In [24]:
# SOLICITUD PARA LA CREACIÓN DE UNA INVITACIÓN DE CONEXIÓN

# Creación de un diccionario JSON con información básica sobre el holder (el que recibirá la credencial)
temp_conn_data = {
    "their_label": temp_wallet.wallet_name,
    "their_role": "holder"
}
print("Info del holder: ", temp_conn_data)
# Apunta al endpoint del agente emisor
temp_conn_url = f"{temp_issuer_client.admin_url}/connections/create-invitation"
print("Endpoint del agente emisor: ", temp_conn_url)
# Se hace un POST con la info básica del holder
temp_conn_response = requests.post(temp_conn_url, json=temp_conn_data, headers=temp_issuer_client.headers)
temp_conn_response.raise_for_status()
temp_connection = temp_conn_response.json()
print("Conexion: ", temp_connection)
# connection_id es el identificador local de la conexión
temp_connection_id = temp_connection["connection_id"]
print("Identificador local de la conexion: ", temp_connection_id)

Info del holder:  {'their_label': 'wallet_6054f6a0', 'their_role': 'holder'}
Endpoint del agente emisor:  http://localhost:9031/connections/create-invitation
Conexion:  {'connection_id': 'fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2', 'invitation': {'@type': 'https://didcomm.org/connections/1.0/invitation', '@id': '8fcc1b24-7c7b-4b73-ad2f-c6fe4d9c6739', 'label': 'Agent with Local Genesis', 'recipientKeys': ['DXBms6fmhr7DrhArPFnPQxw7a5g5LLsnCyurmj3gBKWF'], 'serviceEndpoint': 'http://localhost:9000'}, 'invitation_url': 'http://localhost:9000?c_i=eyJAdHlwZSI6ICJodHRwczovL2RpZGNvbW0ub3JnL2Nvbm5lY3Rpb25zLzEuMC9pbnZpdGF0aW9uIiwgIkBpZCI6ICI4ZmNjMWIyNC03YzdiLTRiNzMtYWQyZi1jNmZlNGQ5YzY3MzkiLCAibGFiZWwiOiAiQWdlbnQgd2l0aCBMb2NhbCBHZW5lc2lzIiwgInJlY2lwaWVudEtleXMiOiBbIkRYQm1zNmZtaHI3RHJoQXJQRm5QUXh3N2E1ZzVMTHNuQ3l1cm1qM2dCS1dGIl0sICJzZXJ2aWNlRW5kcG9pbnQiOiAiaHR0cDovL2xvY2FsaG9zdDo5MDAwIn0'}
Identificador local de la conexion:  fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2


In [25]:
# HOLDER
temp_holder_client = temp_wallet.client

temp_receive_url = f"{temp_holder_client.admin_url}/connections/receive-invitation"
print("Endpoint del holder que recibe la invitación: ", temp_receive_url)
temp_holder_resp = requests.post(
    temp_receive_url,
    json=temp_connection["invitation"],
    headers=temp_holder_client.headers
)
print(temp_holder_resp.json())
temp_holder_conn_id = temp_holder_resp.json()["connection_id"]
print("Connection ID del holder:", temp_holder_conn_id)

Endpoint del holder que recibe la invitación:  http://localhost:9031/connections/receive-invitation
{'state': 'request', 'created_at': '2025-12-05T10:22:55.331071Z', 'updated_at': '2025-12-05T10:22:55.345740Z', 'connection_id': '1d8fe78e-64bc-4e9f-b037-a3f0681180c1', 'my_did': 'Mx6ibwXJ3mmMsKkVrpvsWL', 'their_label': 'Agent with Local Genesis', 'their_role': 'inviter', 'connection_protocol': 'connections/1.0', 'rfc23_state': 'request-sent', 'invitation_key': 'DXBms6fmhr7DrhArPFnPQxw7a5g5LLsnCyurmj3gBKWF', 'invitation_msg_id': '8fcc1b24-7c7b-4b73-ad2f-c6fe4d9c6739', 'request_id': '3a8e2ccc-2651-4535-9226-b683608f5cf2', 'accept': 'auto', 'invitation_mode': 'once'}
Connection ID del holder: 1d8fe78e-64bc-4e9f-b037-a3f0681180c1


In [26]:
# Aquí podemos ver que el issuer ya recibió el connection request del holder y envió la connection response
# Esto es debido a que el agente esta configurado como aceept auto, esto se puede verificar en:
# La línea 53 y 54 de /SSI_App/aries_client/docker-compose.yml

# ISSUER
temp_issuer_check_url = f"{temp_issuer_client.admin_url}/connections/{temp_connection_id}"
print("URL de la aceptación de solicitud del issuer: ", temp_issuer_check_url)
print(requests.get(temp_issuer_check_url, headers=temp_issuer_client.headers).json())

URL de la aceptación de solicitud del issuer:  http://localhost:9031/connections/fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2
{'state': 'invitation', 'created_at': '2025-12-05T10:22:55.317158Z', 'updated_at': '2025-12-05T10:22:55.317158Z', 'connection_id': 'fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2', 'their_role': 'invitee', 'connection_protocol': 'connections/1.0', 'rfc23_state': 'invitation-sent', 'invitation_key': 'DXBms6fmhr7DrhArPFnPQxw7a5g5LLsnCyurmj3gBKWF', 'accept': 'auto', 'invitation_mode': 'once'}


In [27]:
# HOLDER
# En este caso la conexión debería verse completed, pero no se ve porque se está usando connections/1.0
# donde el estado final es response
temp_holder_check_url = f"{temp_holder_client.admin_url}/connections/{temp_holder_conn_id}"
print("URL del estado del holder:", temp_holder_check_url)
print(requests.get(temp_holder_check_url, headers=temp_holder_client.headers).json())

URL del estado del holder: http://localhost:9031/connections/1d8fe78e-64bc-4e9f-b037-a3f0681180c1
{'state': 'request', 'created_at': '2025-12-05T10:22:55.331071Z', 'updated_at': '2025-12-05T10:22:55.345740Z', 'connection_id': '1d8fe78e-64bc-4e9f-b037-a3f0681180c1', 'my_did': 'Mx6ibwXJ3mmMsKkVrpvsWL', 'their_label': 'Agent with Local Genesis', 'their_role': 'inviter', 'connection_protocol': 'connections/1.0', 'rfc23_state': 'request-sent', 'invitation_key': 'DXBms6fmhr7DrhArPFnPQxw7a5g5LLsnCyurmj3gBKWF', 'invitation_msg_id': '8fcc1b24-7c7b-4b73-ad2f-c6fe4d9c6739', 'request_id': '3a8e2ccc-2651-4535-9226-b683608f5cf2', 'accept': 'auto', 'invitation_mode': 'once'}


In [28]:
# CONSTRUCCIÓN DE LA CREDENCIAL
temp_credential_data = {
    "connection_id": temp_connection_id,
    "credential_preview": {
        # Esta sección del mensaje DIDComm sirve para mostrar o anunciar los atributos del credencial antes de emitirlo formalmente
        "@type": "https://didcomm.org/issue-credential/2.0/credential-preview",
        "attributes": temp_attributes
    },
    "filter": {
        "indy": {
            "schema_id": temp_schema_id,
            "cred_def_id": temp_cred_def_id
        }
    },
    "auto_remove": False,
    "trace": False
}
temp_credential_data

{'connection_id': 'fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2',
 'credential_preview': {'@type': 'https://didcomm.org/issue-credential/2.0/credential-preview',
  'attributes': [{'name': 'id_puerto', 'value': 'BER-001'},
   {'name': 'updates', 'value': '0'},
   {'name': 'status', 'value': 'active'}]},
 'filter': {'indy': {'schema_id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:3.0',
   'cred_def_id': 'V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default'}},
 'auto_remove': False,
 'trace': False}

In [29]:
# ANTES DE LA EMISIÓN
# Probamos que la conexión del Issuer está lista
requests.get(f"{temp_issuer_client.admin_url}/connections/{temp_connection_id}", headers=temp_issuer_client.headers).json()

{'state': 'invitation',
 'created_at': '2025-12-05T10:22:55.317158Z',
 'updated_at': '2025-12-05T10:22:55.317158Z',
 'connection_id': 'fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2',
 'their_role': 'invitee',
 'connection_protocol': 'connections/1.0',
 'rfc23_state': 'invitation-sent',
 'invitation_key': 'DXBms6fmhr7DrhArPFnPQxw7a5g5LLsnCyurmj3gBKWF',
 'accept': 'auto',
 'invitation_mode': 'once'}

In [30]:
# Probamos que la definición de credencial existe
requests.get(f"{temp_issuer_client.admin_url}/credential-definitions/created", headers=temp_issuer_client.headers).json()

{'credential_definition_ids': ['V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default',
  'V4SGRU86Z58d6TV7PBUe6f:3:CL:9:default',
  'V4SGRU86Z58d6TV7PBUe6f:3:CL:11:default']}

In [31]:
# PRUEBA DE EMISIÓN (POR FAVOR DIOSITO)
# ISSUER
temp_issuer_cred_url = f"{temp_issuer_client.admin_url}/issue-credential-2.0/send"

# Envía una solicitud HTTP POST al endpoint con todos los datos de la credencial
temp_issuer_cred_response = requests.post(
    temp_issuer_cred_url,
    json=temp_credential_data,
    headers=temp_issuer_client.headers
)

print("Status:", temp_issuer_cred_response.status_code)
print(temp_issuer_cred_response.json())

Status: 200
{'state': 'offer-sent', 'created_at': '2025-12-05T10:23:36.461879Z', 'updated_at': '2025-12-05T10:23:36.461879Z', 'trace': False, 'cred_ex_id': '2ed81d97-9848-4997-82db-36c6c061aad6', 'connection_id': 'fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2', 'thread_id': 'a6b0d487-8fc4-43d7-81a7-14e44f2674f6', 'initiator': 'self', 'role': 'issuer', 'cred_preview': {'@type': 'https://didcomm.org/issue-credential/2.0/credential-preview', 'attributes': [{'name': 'id_puerto', 'value': 'BER-001'}, {'name': 'updates', 'value': '0'}, {'name': 'status', 'value': 'active'}]}, 'cred_proposal': {'@type': 'https://didcomm.org/issue-credential/2.0/propose-credential', '@id': 'a07ddd9b-987b-4006-b0d6-b09fc2f46e27', 'credential_preview': {'@type': 'https://didcomm.org/issue-credential/2.0/credential-preview', 'attributes': [{'name': 'id_puerto', 'value': 'BER-001'}, {'name': 'updates', 'value': '0'}, {'name': 'status', 'value': 'active'}]}, 'formats': [{'attach_id': 'indy', 'format': 'hlindy/cred-filter@v2

In [32]:
# HOLDER
# Revisamos las ofertas que recibio el holder
holder_offers = requests.get(
    f"{temp_holder_client.admin_url}/issue-credential-2.0/records",
    headers=temp_holder_client.headers
).json()

holder_offers

{'results': [{'cred_ex_record': {'state': 'offer-received',
    'created_at': '2025-12-05T10:23:36.489874Z',
    'updated_at': '2025-12-05T10:23:36.489874Z',
    'trace': False,
    'cred_ex_id': '085f8f00-81e5-472b-804a-d8e35e9becc1',
    'connection_id': '1d8fe78e-64bc-4e9f-b037-a3f0681180c1',
    'thread_id': 'a6b0d487-8fc4-43d7-81a7-14e44f2674f6',
    'initiator': 'external',
    'role': 'holder',
    'cred_offer': {'@type': 'https://didcomm.org/issue-credential/2.0/offer-credential',
     '@id': 'a6b0d487-8fc4-43d7-81a7-14e44f2674f6',
     '~thread': {},
     'comment': 'create automated v2.0 credential exchange record',
     'credential_preview': {'@type': 'https://didcomm.org/issue-credential/2.0/credential-preview',
      'attributes': [{'name': 'id_puerto', 'value': 'BER-001'},
       {'name': 'updates', 'value': '0'},
       {'name': 'status', 'value': 'active'}]},
     'formats': [{'attach_id': 'indy', 'format': 'hlindy/cred-abstract@v2.0'}],
     'offers~attach': [{'@id': '

In [33]:
# Obtenemos el cred_ex_id del holder y verificamos que es diferente al del issuer
temp_issuer_cred_json = temp_issuer_cred_response.json()
temp_issuer_cred_ex_id = temp_issuer_cred_json["cred_ex_id"]
temp_holder_cred_ex_id = None

for rec in holder_offers["results"]:
    # Si la clave esta dentro del subregistro 'cred_ex_record'
    if "cred_ex_record" in rec:
        record = rec["cred_ex_record"]
    else:
        record = rec

    if "state" in record and record["state"] == "offer-received":
        temp_holder_cred_ex_id = record["cred_ex_id"]
        break

print("Holder cred_ex_id:", temp_holder_cred_ex_id)
print("Issuer cred_ex_id:", temp_issuer_cred_ex_id)

Holder cred_ex_id: 085f8f00-81e5-472b-804a-d8e35e9becc1
Issuer cred_ex_id: 2ed81d97-9848-4997-82db-36c6c061aad6


In [34]:
# HOLDER
# Logramos revisar que el holder recibio la oferta ahora toca aceptarla
temp_holder_request_url = f"{temp_holder_client.admin_url}/issue-credential-2.0/records/{temp_holder_cred_ex_id}/send-request"
print("URL para aceptar la oferta:", temp_holder_request_url)
temp_holder_request = requests.post(
    temp_holder_request_url,
    headers=temp_holder_client.headers
)

print("RAW RESPONSE:")
print(temp_holder_request.text)

URL para aceptar la oferta: http://localhost:9031/issue-credential-2.0/records/085f8f00-81e5-472b-804a-d8e35e9becc1/send-request
RAW RESPONSE:
{"state": "request-sent", "created_at": "2025-12-05T10:23:36.489874Z", "updated_at": "2025-12-05T10:24:11.167866Z", "trace": false, "cred_ex_id": "085f8f00-81e5-472b-804a-d8e35e9becc1", "connection_id": "1d8fe78e-64bc-4e9f-b037-a3f0681180c1", "thread_id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6", "initiator": "external", "role": "holder", "cred_offer": {"@type": "https://didcomm.org/issue-credential/2.0/offer-credential", "@id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6", "~thread": {}, "comment": "create automated v2.0 credential exchange record", "credential_preview": {"@type": "https://didcomm.org/issue-credential/2.0/credential-preview", "attributes": [{"name": "id_puerto", "value": "BER-001"}, {"name": "updates", "value": "0"}, {"name": "status", "value": "active"}]}, "formats": [{"attach_id": "indy", "format": "hlindy/cred-abstract@v2.0"}], "off

In [35]:
issuer_records = requests.get(f"{temp_issuer_client.admin_url}/issue-credential-2.0/records").json()
print(json.dumps(issuer_records, indent=2))

{
  "results": [
    {
      "cred_ex_record": {
        "state": "credential-issued",
        "created_at": "2025-12-05T10:23:36.461879Z",
        "updated_at": "2025-12-05T10:24:12.275727Z",
        "trace": false,
        "cred_ex_id": "2ed81d97-9848-4997-82db-36c6c061aad6",
        "connection_id": "fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2",
        "thread_id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6",
        "initiator": "self",
        "role": "issuer",
        "cred_preview": {
          "@type": "https://didcomm.org/issue-credential/2.0/credential-preview",
          "attributes": [
            {
              "name": "id_puerto",
              "value": "BER-001"
            },
            {
              "name": "updates",
              "value": "0"
            },
            {
              "name": "status",
              "value": "active"
            }
          ]
        },
        "cred_proposal": {
          "@type": "https://didcomm.org/issue-credential/2.0/propose-credentia

In [36]:
# ISSUER
# Ya que el estado es request-sent ahora toca que el issuer responda generando la credencial y enviandola
temp_issuer_issue_url = f"{temp_issuer_client.admin_url}/issue-credential-2.0/records/{temp_issuer_cred_ex_id}/issue"
print("URL para que el issuer responda:", temp_issuer_issue_url)
temp_issuer_response = requests.post(
    temp_issuer_issue_url,
    headers=temp_issuer_client.headers
)

print("RAW RESPONSE:")
print(temp_issuer_response.text)

# Sin embargo esto da error, porque nuestro issuer esta configurado como auto_issue = true por lo que automáticamente emitio la credencial

URL para que el issuer responda: http://localhost:9031/issue-credential-2.0/records/2ed81d97-9848-4997-82db-36c6c061aad6/issue
RAW RESPONSE:
500 Internal Server Error

Server got itself in trouble


In [37]:
# Verificar el estado del issuer, podemos ver que ya esta en credential_issued
if temp_issuer_cred_ex_id:
    check_url = f"{temp_issuer_client.admin_url}/issue-credential-2.0/records/{temp_issuer_cred_ex_id}"
    check_response = requests.get(check_url, headers=temp_issuer_client.headers)
    print("Estado actual del credential exchange:")
    print(json.dumps(check_response.json(), indent=2))

Estado actual del credential exchange:
{
  "cred_ex_record": {
    "state": "credential-issued",
    "created_at": "2025-12-05T10:23:36.461879Z",
    "updated_at": "2025-12-05T10:24:12.275727Z",
    "trace": false,
    "cred_ex_id": "2ed81d97-9848-4997-82db-36c6c061aad6",
    "connection_id": "fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2",
    "thread_id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6",
    "initiator": "self",
    "role": "issuer",
    "cred_preview": {
      "@type": "https://didcomm.org/issue-credential/2.0/credential-preview",
      "attributes": [
        {
          "name": "id_puerto",
          "value": "BER-001"
        },
        {
          "name": "updates",
          "value": "0"
        },
        {
          "name": "status",
          "value": "active"
        }
      ]
    },
    "cred_proposal": {
      "@type": "https://didcomm.org/issue-credential/2.0/propose-credential",
      "@id": "a07ddd9b-987b-4006-b0d6-b09fc2f46e27",
      "credential_preview": {
        "@t

In [38]:
# Verificar estado del holder
holder_check_url = f"{temp_holder_client.admin_url}/issue-credential-2.0/records/{temp_holder_cred_ex_id}"
holder_check_response = requests.get(holder_check_url, headers=temp_holder_client.headers)
print("Estado actual del holder:")
print(json.dumps(holder_check_response.json(), indent=2))

Estado actual del holder:
{
  "cred_ex_record": {
    "state": "credential-received",
    "created_at": "2025-12-05T10:23:36.489874Z",
    "updated_at": "2025-12-05T10:24:12.291463Z",
    "trace": false,
    "cred_ex_id": "085f8f00-81e5-472b-804a-d8e35e9becc1",
    "connection_id": "1d8fe78e-64bc-4e9f-b037-a3f0681180c1",
    "thread_id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6",
    "initiator": "external",
    "role": "holder",
    "cred_offer": {
      "@type": "https://didcomm.org/issue-credential/2.0/offer-credential",
      "@id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6",
      "~thread": {},
      "comment": "create automated v2.0 credential exchange record",
      "credential_preview": {
        "@type": "https://didcomm.org/issue-credential/2.0/credential-preview",
        "attributes": [
          {
            "name": "id_puerto",
            "value": "BER-001"
          },
          {
            "name": "updates",
            "value": "0"
          },
          {
            "n

In [39]:
# HOLDER
# Almacenar la credencial en el wallet y vemos que esta en estado done
temp_holder_store_url = f"{temp_holder_client.admin_url}/issue-credential-2.0/records/{temp_holder_cred_ex_id}/store"
print("URL para almacenar la credencial:", temp_holder_store_url)

temp_holder_store_response = requests.post(
    temp_holder_store_url,
    headers=temp_holder_client.headers
)

print("Response status:", temp_holder_store_response.status_code)
print("RAW RESPONSE:")
print(temp_holder_store_response.text)

URL para almacenar la credencial: http://localhost:9031/issue-credential-2.0/records/085f8f00-81e5-472b-804a-d8e35e9becc1/store
Response status: 200
RAW RESPONSE:
{"cred_ex_record": {"state": "done", "created_at": "2025-12-05T10:23:36.489874Z", "updated_at": "2025-12-05T10:25:03.634961Z", "trace": false, "cred_ex_id": "085f8f00-81e5-472b-804a-d8e35e9becc1", "connection_id": "1d8fe78e-64bc-4e9f-b037-a3f0681180c1", "thread_id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6", "initiator": "external", "role": "holder", "cred_offer": {"@type": "https://didcomm.org/issue-credential/2.0/offer-credential", "@id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6", "~thread": {}, "comment": "create automated v2.0 credential exchange record", "credential_preview": {"@type": "https://didcomm.org/issue-credential/2.0/credential-preview", "attributes": [{"name": "id_puerto", "value": "BER-001"}, {"name": "updates", "value": "0"}, {"name": "status", "value": "active"}]}, "formats": [{"attach_id": "indy", "format": "hli

In [40]:
# VERIFICACION
# Verificar estado final del holder
# Holder sale con 404 porque tiene auto remove activado, que se puede revisar en la anterior salida, pero vemos que llego al estado done
# por lo que aseguramos que esta bien
print("=== VERIFICACIÓN DEL HOLDER ===")
holder_final_check = requests.get(
    f"{temp_holder_client.admin_url}/issue-credential-2.0/records/{temp_holder_cred_ex_id}",
    headers=temp_holder_client.headers
)

print("Status code:", holder_final_check.status_code)
print("Content type:", holder_final_check.headers.get('content-type'))
print("Primeros 500 caracteres de la respuesta:")
print(holder_final_check.text[:500])

if holder_final_check.status_code == 200:
    try:
        holder_data = holder_final_check.json()
        print("Estado final del holder:")
        print(json.dumps(holder_data, indent=2))
    except json.JSONDecodeError as e:
        print(f"Error decodificando JSON del holder: {e}")
        print("Respuesta completa:")
        print(holder_final_check.text)
else:
    print(f"Error HTTP del holder: {holder_final_check.status_code}")
    print("Respuesta completa:")
    print(holder_final_check.text)

print("\n=== VERIFICACIÓN DEL ISSUER ===")
issuer_final_check = requests.get(
    f"{temp_issuer_client.admin_url}/issue-credential-2.0/records/{temp_issuer_cred_ex_id}",
    headers=temp_issuer_client.headers
)

print("Status code:", issuer_final_check.status_code)
print("Content type:", issuer_final_check.headers.get('content-type'))
print("Primeros 500 caracteres de la respuesta:")
print(issuer_final_check.text[:500])

if issuer_final_check.status_code == 200:
    try:
        issuer_data = issuer_final_check.json()
        print("Estado final del issuer:")
        print(json.dumps(issuer_data, indent=2))
    except json.JSONDecodeError as e:
        print(f"Error decodificando JSON del issuer: {e}")
        print("Respuesta completa:")
        print(issuer_final_check.text)
else:
    print(f"Error HTTP del issuer: {issuer_final_check.status_code}")
    print("Respuesta completa:")
    print(issuer_final_check.text)

=== VERIFICACIÓN DEL HOLDER ===
Status code: 404
Content type: text/plain; charset=utf-8
Primeros 500 caracteres de la respuesta:
404: Record not found: cred_ex_v20/085f8f00-81e5-472b-804a-d8e35e9becc1.
Error HTTP del holder: 404
Respuesta completa:
404: Record not found: cred_ex_v20/085f8f00-81e5-472b-804a-d8e35e9becc1.

=== VERIFICACIÓN DEL ISSUER ===
Status code: 200
Content type: application/json; charset=utf-8
Primeros 500 caracteres de la respuesta:
{"cred_ex_record": {"state": "done", "created_at": "2025-12-05T10:23:36.461879Z", "updated_at": "2025-12-05T10:25:04.684225Z", "trace": false, "cred_ex_id": "2ed81d97-9848-4997-82db-36c6c061aad6", "connection_id": "fa85847c-0f8e-45b1-9a1d-95f2fe1f27a2", "thread_id": "a6b0d487-8fc4-43d7-81a7-14e44f2674f6", "initiator": "self", "role": "issuer", "cred_preview": {"@type": "https://didcomm.org/issue-credential/2.0/credential-preview", "attributes": [{"name": "id_puerto", "value": "BER-001"}, {"name"
Estado final del issuer:
{
  "cred_ex_r

In [41]:
# Verificar credenciales en el wallet del holder
holder_credentials = requests.get(
    f"{temp_holder_client.admin_url}/credentials",
    headers=temp_holder_client.headers
)
print("Credenciales en el wallet del holder:")
print(json.dumps(holder_credentials.json(), indent=2))

Credenciales en el wallet del holder:
{
  "results": [
    {
      "referent": "f5e8fa91-abd5-4ea9-8211-aed3487154b7",
      "schema_id": "V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:3.0",
      "cred_def_id": "V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default",
      "rev_reg_id": null,
      "cred_rev_id": null,
      "attrs": {
        "id_puerto": "BER-001",
        "status": "active",
        "updates": "0"
      }
    }
  ]
}


In [42]:
# VERIFICACIÓN FINAL DE CREDENCIALES EN EL WALLET
print("=== VERIFICACIÓN FINAL - CREDENCIALES EN WALLET DEL HOLDER ===")
holder_credentials = requests.get(
    f"{temp_holder_client.admin_url}/credentials",
    headers=temp_holder_client.headers
)

print("Status code:", holder_credentials.status_code)
if holder_credentials.status_code == 200:
    credentials_data = holder_credentials.json()
    credenciales = credentials_data.get('results', [])
    
    print(f"Número total de credenciales en el wallet: {len(credenciales)}")
    
    if len(credenciales) > 0:
        print("\n=== DETALLE DE CREDENCIALES ENCONTRADAS ===")
        for i, cred in enumerate(credenciales):
            print(f"\n--- Credencial #{i+1} ---")
            print(f"Credential ID: {cred.get('referent')}")
            print(f"Estado: {cred.get('state')}")
            print(f"Schema ID: {cred.get('schema_id')}")
            print(f"Credential Definition ID: {cred.get('cred_def_id')}")
            print(f"Atributos: {json.dumps(cred.get('attrs', {}), indent=2)}")
    else:
        print("❌ No se encontraron credenciales en el wallet")
        
else:
    print(f"❌ Error al obtener credenciales: {holder_credentials.status_code}")
    print("Respuesta:", holder_credentials.text)

# También podemos verificar las conexiones activas
print("\n" + "="*50)
print("=== VERIFICACIÓN DE CONEXIONES ===")
holder_connections = requests.get(
    f"{temp_holder_client.admin_url}/connections",
    headers=temp_holder_client.headers
)

if holder_connections.status_code == 200:
    connections_data = holder_connections.json()
    conexiones = connections_data.get('results', [])
    print(f"Número de conexiones: {len(conexiones)}")
    
    for conn in conexiones:
        if conn.get('state') == 'active':
            print(f"✅ Conexión activa: {conn.get('connection_id')} con {conn.get('their_label', 'Unknown')}")
else:
    print(f"Error al obtener conexiones: {holder_connections.status_code}")

=== VERIFICACIÓN FINAL - CREDENCIALES EN WALLET DEL HOLDER ===
Status code: 200
Número total de credenciales en el wallet: 1

=== DETALLE DE CREDENCIALES ENCONTRADAS ===

--- Credencial #1 ---
Credential ID: f5e8fa91-abd5-4ea9-8211-aed3487154b7
Estado: None
Schema ID: V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:3.0
Credential Definition ID: V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default
Atributos: {
  "updates": "0",
  "status": "active",
  "id_puerto": "BER-001"
}

=== VERIFICACIÓN DE CONEXIONES ===
Número de conexiones: 1
✅ Conexión activa: 1d8fe78e-64bc-4e9f-b037-a3f0681180c1 con Agent with Local Genesis


In [ ]:
from client import ACApyClient

In [4]:
client = ACApyClient()
credencial = client.register_credential("Evtol_Cred", "1.0", ["id_puerto", "updates", "status"])
credencial

{'sent': {'schema_id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Cred:1.0',
  'schema': {'ver': '1.0',
   'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Cred:1.0',
   'name': 'Evtol_Cred',
   'version': '1.0',
   'attrNames': ['id_puerto', 'updates', 'status'],
   'seqNo': 15}},
 'schema_id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Cred:1.0',
 'schema': {'ver': '1.0',
  'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Cred:1.0',
  'name': 'Evtol_Cred',
  'version': '1.0',
  'attrNames': ['id_puerto', 'updates', 'status'],
  'seqNo': 15}}

In [2]:
client = ACApyClient()
credential_def = client.create_credential_definition('V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Cred:1.0')
credential_def

{'sent': {'credential_definition_id': 'V4SGRU86Z58d6TV7PBUe6f:3:CL:15:default'},
 'credential_definition_id': 'V4SGRU86Z58d6TV7PBUe6f:3:CL:15:default'}

In [3]:
credential_def['credential_definition_id']

'V4SGRU86Z58d6TV7PBUe6f:3:CL:15:default'

In [9]:
credencial['schema']

{'ver': '1.0',
 'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Cred:1.0',
 'name': 'Evtol_Cred',
 'version': '1.0',
 'attrNames': ['id_puerto', 'updates', 'status'],
 'seqNo': 15}

In [ ]:
from client import ACApyClient
client = ACApyClient()

In [4]:
invitacion = client.create_invitation()
invitacion

{'connection_id': 'edc2e3d1-9606-47ab-aa0b-a7cda63c7f97',
 'invitation': {'@type': 'https://didcomm.org/connections/1.0/invitation',
  '@id': '9fa3f991-963a-4082-aa61-037382457a7c',
  'label': 'Agent with Local Genesis',
  'recipientKeys': ['6XQweSa74esd4zStuYX3rx34fuVyGhEXLPnn5UFBqPyG'],
  'serviceEndpoint': 'http://localhost:3000'},
 'invitation_url': 'http://localhost:3000?c_i=eyJAdHlwZSI6ICJodHRwczovL2RpZGNvbW0ub3JnL2Nvbm5lY3Rpb25zLzEuMC9pbnZpdGF0aW9uIiwgIkBpZCI6ICI5ZmEzZjk5MS05NjNhLTQwODItYWE2MS0wMzczODI0NTdhN2MiLCAibGFiZWwiOiAiQWdlbnQgd2l0aCBMb2NhbCBHZW5lc2lzIiwgInJlY2lwaWVudEtleXMiOiBbIjZYUXdlU2E3NGVzZDR6U3R1WVgzcngzNGZ1VnlHaEVYTFBubjVVRkJxUHlHIl0sICJzZXJ2aWNlRW5kcG9pbnQiOiAiaHR0cDovL2xvY2FsaG9zdDozMDAwIn0'}

In [5]:
invitacion['connection_id']

'edc2e3d1-9606-47ab-aa0b-a7cda63c7f97'

In [ ]:
from client import ACApyClient

client = ACApyClient()

In [2]:
schema_body = [
    {"name":"id_puerto", "value": "puerto_1"}, 
    {"name":"updates", "value": "1"}, 
    {"name":"status", "value": "READY"}
]
schema_body

[{'name': 'id_puerto', 'value': 'puerto_1'},
 {'name': 'updates', 'value': '1'},
 {'name': 'status', 'value': 'READY'}]

In [7]:
credencial['schema_id']

'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Cred:1.0'

In [5]:
response = client.send_credential_offer('V4SGRU86Z58d6TV7PBUe6f:3:CL:15:default',schema_body)
response

📤 Enviando oferta de credencial: {
  "connection_id": "a8668a5f-ae3a-49b9-a8b8-470271143ab2",
  "cred_def_id": "V4SGRU86Z58d6TV7PBUe6f:3:CL:15:default",
  "credential_preview": {
    "@type": "issue-credential/1.0/credential-preview",
    "attributes": [
      {
        "name": "id_puerto",
        "value": "puerto_1"
      },
      {
        "name": "updates",
        "value": "1"
      },
      {
        "name": "status",
        "value": "READY"
      }
    ]
  },
  "auto_issue": true,
  "auto_remove": true
}
📥 Status: 403
📥 Body: 403: Connection a8668a5f-ae3a-49b9-a8b8-470271143ab2 not ready


{'error': 'HTTP 403',
 'raw_response': '403: Connection a8668a5f-ae3a-49b9-a8b8-470271143ab2 not ready'}

In [20]:
test_e = Evtol()

In [21]:
test_e.wallet.wallet_data

{'result': {'did': 'SLLmPu4kQHpkkADojXTDkT',
  'verkey': 'Eoqm1bKwugGuUGpo9TAMEWSAWGNnojiAz55d59Ar4PNe',
  'posture': 'wallet_only',
  'key_type': 'ed25519',
  'method': 'sov',
  'metadata': {}}}

In [17]:
test_e.wallet.wallet_data

{'result': {'did': 'J3beHsH91DtrPUENVjHSmD',
  'verkey': 'AHpWYKyFS964suX3BAS8vHajxv2FaTxm7FYM6i7GJ6vm',
  'posture': 'wallet_only',
  'key_type': 'ed25519',
  'method': 'sov',
  'metadata': {}}}

In [19]:
test_e3 = Evtol(id="EVTOL-12345")
test_e3.wallet.wallet_data

{'result': {'did': 'FzqNesD326ecbHACFMuc24',
  'verkey': '9B6Akxu3XvfJ3U6qyr6L1yJfCPwoypkTqZ5aoRvQSHHY',
  'posture': 'wallet_only',
  'key_type': 'ed25519',
  'method': 'sov',
  'metadata': {}}}